# What does a large amount of Icelandic króna mean?

This prototype translates an amount in ISK into several scales using current Statistics Iceland tables. It distinguishes between:

- **Public-service spending equivalents** — months at the 2025 annual-average spending rate.
- **Economic-flow equivalents** — days at the 2025 annual-average rate.
- **People-scale equivalents** — per resident, primary-school student-years, and full-time earnings.
- **Editable assumptions** — familiar objects such as cars, whose retail price is not supplied cleanly by the API.

These are scale comparisons, not claims that money can literally be transferred between categories without changing outcomes.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import nordicstats as ns

pd.set_option("display.max_colwidth", 120)
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
print("Python package version:", ns.__version__)
print("Project root:", PROJECT_ROOT)

## 1. Enter the amount

Change `amount_isk` and rerun the cells below. `assumed_car_price_isk` is deliberately separate: it is a user-editable assumption, not an official Statistics Iceland price.

In [ ]:
amount_isk = 23_000_000_000
assumed_car_price_isk = 8_000_000

def format_isk(value: float) -> str:
    if abs(value) >= 1_000_000_000:
        return f"{value / 1_000_000_000:,.1f} billion ISK"
    if abs(value) >= 1_000_000:
        return f"{value / 1_000_000:,.1f} million ISK"
    return f"{value:,.0f} ISK"

print("Amount to explain:", format_isk(amount_isk))

## 2. Fetch small, explicit API selections

Every category code below was inspected in the live metadata. Values in public-finance and national-account tables are converted from **million ISK** to ISK. The fish-catch table already returns ISK, student cost returns ISK, and earnings return thousand ISK per month.

In [ ]:
ice = ns.client_for("iceland")

TABLES = {
    "population": "Ibuar/mannfjoldi/1_yfirlit/Yfirlit_mannfjolda/MAN00101.px",
    "health": "Efnahagur/fjaropinber/fjarmal_heilbr/THJ05511.px",
    "education": "Efnahagur/fjaropinber/fjarmal_fraedsla/1_utgjold_fraedsla/THJ05611.px",
    "government_functions": "Efnahagur/fjaropinber/fjarmal_opinber/fjarmal_opinber/THJ05142.px",
    "student_cost": "Efnahagur/fjaropinber/fjarmal_fraedsla/1_utgjold_fraedsla/THJ05636A.px",
    "gdp": "Efnahagur/thjodhagsreikningar/landsframl/1_landsframleidsla/THJ01102.px",
    "household_consumption": "Efnahagur/thjodhagsreikningar/einkaneysla/THJ02103.px",
    "tourism": "Efnahagur/thjodhagsreikningar/ferdathjonustureikningar/ferdathjonustureikningar/SAM08000.px",
    "earnings": "Samfelag/launogtekjur/1_laun/1_laun/VIN02002.px",
    "fish_catch": "Atvinnuvegir/sjavarutvegur/aflatolur/afli_verdmaeti/SJA02201.px",
}

def fetch_one(table_key: str, selection: dict, multiplier: float = 1.0) -> float:
    frame = ice.fetch(TABLES[table_key], selection)
    if len(frame) != 1:
        raise ValueError(f"Expected one value from {table_key}, received {len(frame)}")
    value = frame["value"].iloc[0]
    if pd.isna(value):
        raise ValueError(f"The selected value for {table_key} is missing")
    return float(value) * multiplier

In [ ]:
MILLION_ISK = 1_000_000
THOUSAND_ISK = 1_000

population_2025 = fetch_one(
    "population", {"Kyn": ["Total"], "Aldur": ["-1"], "Ár": ["2025"]}
)
health_2025 = fetch_one(
    "health", {"Eining": ["0"], "Skipting": ["14"], "Ár": ["2025"]}, MILLION_ISK
)
education_2025 = fetch_one(
    "education", {"Eining": ["0"], "Skipting": ["11"], "Ár": ["2025"]}, MILLION_ISK
)
student_cost_2025 = fetch_one(
    "student_cost", {"Ár": ["2025"]}
)
gdp_2025 = fetch_one(
    "gdp", {"Skipting": ["11"], "Ár": ["2025"]}, MILLION_ISK
)
tourism_2025 = fetch_one(
    "tourism", {"Ár": ["2025"], "Atvinnugrein": ["0"], "Ferðamenn": ["2"]}, MILLION_ISK
)
mean_monthly_earnings_2025 = fetch_one(
    "earnings",
    {"Ár": ["2025"], "Launþegahópur": ["0"], "Starfsstétt": ["0"], "Kyn": ["0"],
     "Laun og vinnutími": ["7"], "Eining": ["0"]},
    THOUSAND_ISK,
)
fish_value_2025 = fetch_one(
    "fish_catch",
    {"Species": ["0"], "Fishing area": ["0"], "Unit": ["1"], "Year": ["15"], "Month": ["0"]},
)

government = ice.fetch(
    TABLES["government_functions"],
    {"Eining": ["Million ISK"], "Skipting": ["03", "06", "08", "10"], "Ár": ["2025"]},
    labels=True,
)
government_values = dict(zip(government["Skipting"], government["value"] * MILLION_ISK))

household = ice.fetch(
    TABLES["household_consumption"],
    {"Mælikvarði": ["0"], "Skipting": ["0", "1", "10"], "Ár": ["2025"]},
    labels=True,
)
household_values = dict(zip(household["Skipting"], household["value"] * MILLION_ISK))

print("Fetched reference values successfully.")

## 3. People-scale comparisons

These cards are easiest to understand, but each has a different interpretation. Earnings are gross monthly earnings, not the total employer cost. Student cost is average operational expenditure, not the marginal cost of adding one pupil.

In [ ]:
people_scale = pd.DataFrame(
    [
        {"comparison": "Per resident of Iceland", "equivalent": amount_isk / population_2025, "unit": "ISK per resident", "basis": "2025 population"},
        {"comparison": "Primary-school operational cost", "equivalent": amount_isk / student_cost_2025, "unit": "student-years", "basis": "2025 average cost"},
        {"comparison": "Mean full-time earnings", "equivalent": amount_isk / mean_monthly_earnings_2025 / 12, "unit": "worker-years of gross earnings", "basis": "2025 survey mean"},
        {"comparison": "Cars (editable assumption)", "equivalent": amount_isk / assumed_car_price_isk, "unit": "cars", "basis": f"assumed {format_isk(assumed_car_price_isk)} each"},
    ]
)
people_scale["display"] = people_scale.apply(
    lambda row: f"{row['equivalent']:,.0f} {row['unit']}", axis=1
)
display(people_scale[["comparison", "display", "basis"]])

## 4. Public-service scale

Read each bar as: **the amount equals approximately this many months of 2025 expenditure at an annual-average rate**. This is more accurate than saying the money would literally “run” a service for that period.

In [ ]:
service_annual = {
    "Public health": health_2025,
    "Public education": education_2025,
    "Public order and safety": government_values["03 Public order and safety"],
    "Housing and community amenities": government_values["06 Housing and community amenities"],
    "Recreation, culture and religion": government_values["08 Recreation, culture and religion"],
    "Social protection": government_values["10 Social protection"],
}
service_scale = pd.DataFrame(
    {"category": service_annual.keys(), "annual_isk": service_annual.values()}
)
service_scale["equivalent_months"] = amount_isk / service_scale["annual_isk"] * 12
service_scale = service_scale.sort_values("equivalent_months")

fig, ax = plt.subplots(figsize=(10, 5.5))
bars = ax.barh(service_scale["category"], service_scale["equivalent_months"], color="#3B82A0")
ax.bar_label(bars, labels=[f"{value:.2f} months" for value in service_scale["equivalent_months"]], padding=4)
ax.set_title(f"{format_isk(amount_isk)} in public-service scale", loc="left", weight="bold")
ax.set_xlabel("Equivalent months at the 2025 annual-average expenditure rate")
ax.spines[["top", "right", "left"]].set_visible(False)
ax.grid(axis="x", alpha=0.2)
plt.tight_layout()
plt.show()

## 5. National economic-flow scale

These bars compare the amount with annual flows. “GDP days” means the amount is the same fraction of annual GDP as that number of average days—not that GDP can be purchased. Tourism expenditure for 2025 is preliminary, and fish-catch value may be revised.

In [ ]:
flow_annual = {
    "Gross domestic product": gdp_2025,
    "All household consumption": household_values["Total"],
    "Household food and non-alcoholic drinks": household_values["01. Food and non-alcoholic beverages"],
    "Household housing and utilities": household_values["04. Housing, water, electricity, gas and other fuels"],
    "Inbound visitor expenditure": tourism_2025,
    "Value of Icelandic fish catch": fish_value_2025,
}
flow_scale = pd.DataFrame({"category": flow_annual.keys(), "annual_isk": flow_annual.values()})
flow_scale["equivalent_days"] = amount_isk / flow_scale["annual_isk"] * 365
flow_scale = flow_scale.sort_values("equivalent_days")

fig, ax = plt.subplots(figsize=(10, 5.5))
bars = ax.barh(flow_scale["category"], flow_scale["equivalent_days"], color="#D97757")
ax.bar_label(bars, labels=[f"{value:.1f} days" for value in flow_scale["equivalent_days"]], padding=4)
ax.set_title(f"{format_isk(amount_isk)} in national economic-flow scale", loc="left", weight="bold")
ax.set_xlabel("Equivalent days at the 2025 annual-average rate")
ax.spines[["top", "right", "left"]].set_visible(False)
ax.grid(axis="x", alpha=0.2)
plt.tight_layout()
plt.show()

## 6. Sources, caveats, and optional export

All official comparisons use Statistics Iceland’s PxWeb API and 2025 reference data. Current-price values are used so they are comparable with a current nominal amount. Important caveats should remain visible in a future website:

- Results are equivalences of scale, not budget proposals or forecasts.
- Annual flows are divided evenly across 365 days or 12 months for communication; real activity is seasonal.
- 2025 GDP and tourism figures are preliminary. Fish-catch figures may be revised.
- Average earnings are gross earnings from a sample survey, not employer cost.
- The car result is based only on the editable assumption above.

Official table pages: [health expenditure](https://px.hagstofa.is/pxen/pxweb/en/Efnahagur/Efnahagur__fjaropinber__fjarmal_heilbr/THJ05511.px), [government expenditure by function](https://px.hagstofa.is/pxen/pxweb/en/Efnahagur/Efnahagur__fjaropinber__fjarmal_opinber__fjarmal_opinber/THJ05142.px), [GDP](https://px.hagstofa.is/pxen/pxweb/en/Efnahagur/Efnahagur__thjodhagsreikningar__landsframl__1_landsframleidsla/THJ01102.px), [household consumption](https://px.hagstofa.is/pxen/pxweb/en/Efnahagur/Efnahagur__thjodhagsreikningar__einkaneysla/THJ02103.px), [tourism expenditure](https://px.hagstofa.is/pxen/pxweb/en/Efnahagur/Efnahagur__thjodhagsreikningar__ferdathjonustureikningar__ferdathjonustureikningar/SAM08000.px), and [fish-catch value](https://px.hagstofa.is/pxen/pxweb/en/Atvinnuvegir/Atvinnuvegir__sjavarutvegur__aflatolur__afli_verdmaeti/SJA02201.px).

In [ ]:
# Optional reproducible export for later website prototyping.
export = pd.concat(
    [
        service_scale.assign(section="public_service", equivalent_unit="months").rename(columns={"equivalent_months": "equivalent"}),
        flow_scale.assign(section="economic_flow", equivalent_unit="days").rename(columns={"equivalent_days": "equivalent"}),
    ],
    ignore_index=True,
)[['section', 'category', 'annual_isk', 'equivalent', 'equivalent_unit']]
export.insert(0, "input_amount_isk", amount_isk)
display(export)

# Uncomment to save under the project-level data directory.
# output_path = PROJECT_ROOT / "data" / "isk_scale_comparisons.csv"
# output_path.parent.mkdir(exist_ok=True)
# export.to_csv(output_path, index=False)
# print("Saved", output_path)